
## Çok Katmanlı Algılayıcı (MLP) ile Wine Quality Çoklu Sınıflandırma

**Veri Seti:** Wine Quality (UCI ML Repository) - 6497 satır, 11 özellik, 7 sınıf

**İçerik:**
1. Veri Setini Yükleme ve Keşifsel Veri Analizi (EDA)
2. Veri Ön İşleme
3. Model 1: Sıfırdan MLP (1 Gizli Katman) - Softmax Çıkışlı
4. Model 2: Sıfırdan MLP (2 Gizli Katman)
5. Overfitting / Underfitting Analizi
6. Model 3: L2 Regülarizasyonlu MLP
7. Hiperparametre Araması
8. Model 4: Scikit-learn MLPClassifier ile Karşılaştırma
9. Model 5: PyTorch MLP ile Karşılaştırma
10. Tüm Modellerin Karşılaştırılması ve Sonuçlar

---
## 1. Veri Setini Yükleme ve Keşifsel Veri Analizi (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Tekrarlanabilirlik
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Kütüphaneler yüklendi.')
# Çıktı dizinini oluştur
os.makedirs('../results/wine_quality', exist_ok=True)


### 1.1 Veri Setini Yükleme

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import DataLoader

loader = DataLoader(random_state=RANDOM_STATE)
df = loader.load_wine_quality('../data/wine_quality.csv')

### 1.2 Veri Seti Bilgileri

In [ ]:
print('Veri Seti Boyutu:', df.shape)
print('\nSütunlar:', list(df.columns))
print('\nVeri Tipleri:')
print(df.dtypes)
print('\nİlk 10 Satır:')
df.head(10)

In [ ]:
df.info()

In [ ]:
print('İstatistiksel Özet:')
df.describe().round(4)

### 1.3 Eksik Değer Kontrolü

In [ ]:
print('Eksik Değer Sayıları:')
null_counts = df.isnull().sum()
print(null_counts)
print(f'\nToplam eksik değer: {null_counts.sum()}')
if null_counts.sum() == 0:
    print('✅ Eksik değer bulunmamaktadır.')

### 1.4 Sınıf Dağılımı

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
# Orijinal kalite etiketleri (3-9), 0'dan başlayacak şekilde dönüştürüldü
min_q = 3  # Orijinal minimum kalite
class_counts = df['quality'].value_counts().sort_index()

print('Sınıf Dağılımı:')
for cls, cnt in class_counts.items():
    pct = cnt / len(df) * 100
    print(f'  Sınıf {cls} (Kalite {cls + min_q}): {cnt} örnek ({pct:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('viridis', len(class_counts))
bars = ax.bar([f'Kalite {i+min_q}' for i in class_counts.index], class_counts.values, color=colors, edgecolor='black')
ax.set_title('Wine Quality - Sınıf Dağılımı', fontsize=14, fontweight='bold')
ax.set_xlabel('Kalite Sınıfı', fontsize=12)
ax.set_ylabel('Örnek Sayısı', fontsize=12)
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            str(count), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/wine_quality/WineQuality_sinif_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 Korelasyon Matrisi

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            ax=ax, mask=mask, square=True, linewidths=0.5)
ax.set_title('Wine Quality - Korelasyon Matrisi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/wine_quality/WineQuality_korelasyon.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.6 Özellik Dağılımları

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
features = df.columns[:-1]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    for cls in sorted(df['quality'].unique()):
        subset = df[df['quality'] == cls][feat]
        ax.hist(subset, bins=30, alpha=0.5, label=f'Kalite {cls+min_q}', density=True)
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=6)

# Son boş subplot'u gizle
axes[-1].set_visible(False)

fig.suptitle('Wine Quality - Özelliklerin Sınıflara Göre Dağılımı', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/wine_quality/WineQuality_ozellik_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.7 Box Plot

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    df.boxplot(column=feat, by='quality', ax=ax)
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.set_xlabel('Kalite Sınıfı')
    ax.set_ylabel('')

axes[-1].set_visible(False)
fig.suptitle('Wine Quality - Özelliklerin Sınıflara Göre Box Plot', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/wine_quality/WineQuality_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Veri Ön İşleme

### 2.1 Veri Bölme ve Ölçeklendirme

Veri seti **%70 eğitim**, **%15 doğrulama (validation)**, **%15 test** olarak bölünecektir.
Standardizasyon (**StandardScaler**) uygulanacaktır.

In [ ]:
# Veri bölme ve ölçeklendirme
X_train, X_val, X_test, y_train, y_val, y_test = loader.split_data(
    test_size=0.15, val_size=0.15, scaling='standard'
)

n_features = X_train.shape[1]
n_classes = loader.n_classes

print(f'\nÖzellik sayısı (n_features): {n_features}')
print(f'Sınıf sayısı (n_classes): {n_classes}')
print(f'\nX_train shape: {X_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_val shape:   {y_val.shape}')
print(f'y_test shape:  {y_test.shape}')

---
## 3. Model 1: Sıfırdan MLP (1 Gizli Katman) - Softmax Çıkışlı

Lab çalışmasındaki 2 katmanlı (1 gizli katmanlı) model, burada **çoklu sınıflandırma** için
softmax çıkış katmanı ve **categorical cross entropy** kayıp fonksiyonu ile uyarlanmıştır.

**Mimari:** `[11, 32, 7]` (11 giriş → 32 gizli nöron → 7 çıkış)

**Hiperparametreler:**
- Öğrenme oranı (learning rate): 0.01
- Adım sayısı (n_steps): 2000
- Optimizasyon: SGD
- Gizli katman aktivasyonu: tanh
- Çıkış katmanı: softmax

In [ ]:
from src.mlp_scratch import MLPScratch
from src.metrics import print_metrics, plot_confusion_matrix, plot_cost_curves, plot_accuracy_curves

# Model 1: 1 Gizli Katman [11, 32, 7]
model1 = MLPScratch(
    layer_dims=[n_features, 32, n_classes],
    learning_rate=0.01,
    n_steps=2000,
    random_state=RANDOM_STATE,
    lambda_reg=0.0,
    task='multiclass',
    activation_hidden='tanh'
)

print('Model 1 Mimarisi:', model1)
print('\nEğitim başlıyor...')
model1.fit(X_train, y_train, X_val=X_val, y_val=y_val, print_cost=True, print_every=200)

### 3.1 Model 1 - Eğitim ve Doğrulama Eğrileri

In [ ]:
# Cost eğrileri
plot_cost_curves(model1.cost_history_train, model1.cost_history_val,
                 model_name='Model 1 (1 Gizli Katman [11,32,7])',
                 save_path='../results/wine_quality/model1_cost.png')

# Accuracy eğrileri
plot_accuracy_curves(model1.accuracy_history_train, model1.accuracy_history_val,
                     model_name='Model 1 (1 Gizli Katman [11,32,7])',
                     save_path='../results/wine_quality/model1_accuracy.png')

### 3.2 Model 1 - Test Sonuçları

In [ ]:
# Test seti üzerinde değerlendirme
y_pred_m1 = model1.predict(X_test)
results_m1 = print_metrics(y_test, y_pred_m1, model_name='Model 1 (1 Gizli Katman)',
                           task='multiclass')

# Confusion Matrix
plot_confusion_matrix(y_test, y_pred_m1,
                      model_name='Model 1 (1 Gizli Katman [11,32,7])',
                      class_names=[f'K{i+3}' for i in range(n_classes)],
                      save_path='../results/wine_quality/model1_cm.png')

---
## 4. Model 2: Sıfırdan MLP (2 Gizli Katman)

Gizli katman sayısı artırılarak model kapasitesi yükseltilir.

**Mimari:** `[11, 64, 32, 7]` (11 → 64 → 32 → 7)

**Hiperparametreler:**
- Öğrenme oranı: 0.01
- Adım sayısı: 2000
- Optimizasyon: SGD
- Gizli katman aktivasyonu: tanh

In [ ]:
# Model 2: 2 Gizli Katman [11, 64, 32, 7]
model2 = MLPScratch(
    layer_dims=[n_features, 64, 32, n_classes],
    learning_rate=0.01,
    n_steps=2000,
    random_state=RANDOM_STATE,
    lambda_reg=0.0,
    task='multiclass',
    activation_hidden='tanh'
)

print('Model 2 Mimarisi:', model2)
print('\nEğitim başlıyor...')
model2.fit(X_train, y_train, X_val=X_val, y_val=y_val, print_cost=True, print_every=200)

### 4.1 Model 2 - Eğitim ve Doğrulama Eğrileri

In [ ]:
plot_cost_curves(model2.cost_history_train, model2.cost_history_val,
                 model_name='Model 2 (2 Gizli Katman [11,64,32,7])',
                 save_path='../results/wine_quality/model2_cost.png')

plot_accuracy_curves(model2.accuracy_history_train, model2.accuracy_history_val,
                     model_name='Model 2 (2 Gizli Katman [11,64,32,7])',
                     save_path='../results/wine_quality/model2_accuracy.png')

### 4.2 Model 2 - Test Sonuçları

In [ ]:
y_pred_m2 = model2.predict(X_test)
results_m2 = print_metrics(y_test, y_pred_m2, model_name='Model 2 (2 Gizli Katman)',
                           task='multiclass')

plot_confusion_matrix(y_test, y_pred_m2,
                      model_name='Model 2 (2 Gizli Katman [11,64,32,7])',
                      class_names=[f'K{i+3}' for i in range(n_classes)],
                      save_path='../results/wine_quality/model2_cm.png')

---
## 5. Overfitting / Underfitting Analizi

Eğitim ve doğrulama set kayıp (cost) değerlerinin karşılaştırılması yoluyla modellerin
**yüksek varyans** (overfitting) veya **yüksek bias** (underfitting) durumu incelenir.

- **Overfitting belirtisi:** Train cost düşüyor, val cost artıyor (veya sabit kalıyor).
- **Underfitting belirtisi:** Her iki cost da yüksek kalıyor, iyileşme yavaş.
- **Çözümler:** Regülarizasyon, daha fazla veri, model karmaşıklığını azaltma/artırma.

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
# Model 1 ve Model 2 karşılaştırmalı analiz
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Model 1
ax = axes[0]
ax.plot(model1.cost_history_train, label='Eğitim (Train)', color='#2196F3', linewidth=2)
ax.plot(model1.cost_history_val, label='Doğrulama (Val)', color='#FF5722', linewidth=2, linestyle='--')
ax.set_title('Model 1 (1 Gizli Katman)', fontweight='bold')
ax.set_xlabel('Adım (Epoch)')
ax.set_ylabel('Maliyet (Cost)')
ax.legend()
ax.grid(True, alpha=0.3)

# Model 2
ax = axes[1]
ax.plot(model2.cost_history_train, label='Eğitim (Train)', color='#2196F3', linewidth=2)
ax.plot(model2.cost_history_val, label='Doğrulama (Val)', color='#FF5722', linewidth=2, linestyle='--')
ax.set_title('Model 2 (2 Gizli Katman)', fontweight='bold')
ax.set_xlabel('Adım (Epoch)')
ax.set_ylabel('Maliyet (Cost)')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('Overfitting / Underfitting Analizi', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('../results/wine_quality/overfitting_analizi.png', dpi=150, bbox_inches='tight')
plt.show()

# Nihai değerler karşılaştırması
print('\n--- Overfitting/Underfitting Özet ---')
print(f'Model 1: Train Cost={model1.cost_history_train[-1]:.4f}, Val Cost={model1.cost_history_val[-1]:.4f}, '
      f'Train Acc={model1.accuracy_history_train[-1]:.4f}, Val Acc={model1.accuracy_history_val[-1]:.4f}')
print(f'Model 2: Train Cost={model2.cost_history_train[-1]:.4f}, Val Cost={model2.cost_history_val[-1]:.4f}, '
      f'Train Acc={model2.accuracy_history_train[-1]:.4f}, Val Acc={model2.accuracy_history_val[-1]:.4f}')

# Yorum
gap1 = model1.accuracy_history_train[-1] - model1.accuracy_history_val[-1]
gap2 = model2.accuracy_history_train[-1] - model2.accuracy_history_val[-1]
print(f'\nModel 1 Train-Val Accuracy Farkı: {gap1:.4f}')
print(f'Model 2 Train-Val Accuracy Farkı: {gap2:.4f}')
if gap2 > 0.05:
    print('⚠️ Model 2 overfitting eğilimi gösteriyor. Regülarizasyon uygulanmalı.')
else:
    print('✅ Modeller arasında ciddi overfitting gözlenmemiştir.')

---
## 6. Model 3: L2 Regülarizasyonlu MLP

Overfitting'i azaltmak için **L2 regülarizasyon** (weight decay) uygulanır.

**Mimari:** `[11, 64, 32, 7]` (Model 2 ile aynı)

**Ek Hiperparametre:**
- L2 regülarizasyon katsayısı (λ): 0.01

In [ ]:
# Model 3: 2 Gizli Katman + L2 Regülarizasyon
model3 = MLPScratch(
    layer_dims=[n_features, 64, 32, n_classes],
    learning_rate=0.01,
    n_steps=2000,
    random_state=RANDOM_STATE,
    lambda_reg=0.01,
    task='multiclass',
    activation_hidden='tanh'
)

print('Model 3 Mimarisi:', model3)
print('\nEğitim başlıyor...')
model3.fit(X_train, y_train, X_val=X_val, y_val=y_val, print_cost=True, print_every=200)

### 6.1 Model 3 - Eğitim ve Doğrulama Eğrileri

In [ ]:
plot_cost_curves(model3.cost_history_train, model3.cost_history_val,
                 model_name='Model 3 (L2 Regülarizasyon λ=0.01)',
                 save_path='../results/wine_quality/model3_cost.png')

plot_accuracy_curves(model3.accuracy_history_train, model3.accuracy_history_val,
                     model_name='Model 3 (L2 Regülarizasyon λ=0.01)',
                     save_path='../results/wine_quality/model3_accuracy.png')

### 6.2 Model 3 - Test Sonuçları

In [ ]:
y_pred_m3 = model3.predict(X_test)
results_m3 = print_metrics(y_test, y_pred_m3, model_name='Model 3 (L2 Regülarizasyon)',
                           task='multiclass')

plot_confusion_matrix(y_test, y_pred_m3,
                      model_name='Model 3 (L2 Reg. [11,64,32,7] λ=0.01)',
                      class_names=[f'K{i+3}' for i in range(n_classes)],
                      save_path='../results/wine_quality/model3_cm.png')

### 6.3 Regülarizasyon Etkisi Karşılaştırma

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
# Model 2 (regsüz) vs Model 3 (regülarizasyonlu)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(model2.cost_history_train, label='M2 Train (λ=0)', color='#2196F3', linewidth=2)
ax.plot(model2.cost_history_val, label='M2 Val (λ=0)', color='#2196F3', linewidth=2, linestyle='--')
ax.plot(model3.cost_history_train, label='M3 Train (λ=0.01)', color='#E91E63', linewidth=2)
ax.plot(model3.cost_history_val, label='M3 Val (λ=0.01)', color='#E91E63', linewidth=2, linestyle='--')
ax.set_title('Cost Karşılaştırması', fontweight='bold')
ax.set_xlabel('Adım')
ax.set_ylabel('Cost')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(model2.accuracy_history_train, label='M2 Train (λ=0)', color='#2196F3', linewidth=2)
ax.plot(model2.accuracy_history_val, label='M2 Val (λ=0)', color='#2196F3', linewidth=2, linestyle='--')
ax.plot(model3.accuracy_history_train, label='M3 Train (λ=0.01)', color='#E91E63', linewidth=2)
ax.plot(model3.accuracy_history_val, label='M3 Val (λ=0.01)', color='#E91E63', linewidth=2, linestyle='--')
ax.set_title('Accuracy Karşılaştırması', fontweight='bold')
ax.set_xlabel('Adım')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('L2 Regülarizasyon Etkisi', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('../results/wine_quality/regularizasyon_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Hiperparametre Araması

Farklı gizli katman nöron sayıları, öğrenme oranları ve adım sayıları denenerek
en iyi yapılandırma aranır.

In [ ]:
# Hiperparametre grid arama
print('Hiperparametre Araması Başlıyor...\n')

configs = [
    {'dims': [n_features, 16, n_classes], 'lr': 0.01, 'steps': 2000, 'name': '[11,16,7] lr=0.01'},
    {'dims': [n_features, 32, n_classes], 'lr': 0.01, 'steps': 2000, 'name': '[11,32,7] lr=0.01'},
    {'dims': [n_features, 64, n_classes], 'lr': 0.01, 'steps': 2000, 'name': '[11,64,7] lr=0.01'},
    {'dims': [n_features, 32, n_classes], 'lr': 0.05, 'steps': 2000, 'name': '[11,32,7] lr=0.05'},
    {'dims': [n_features, 64, 32, n_classes], 'lr': 0.01, 'steps': 2000, 'name': '[11,64,32,7] lr=0.01'},
    {'dims': [n_features, 64, 32, n_classes], 'lr': 0.05, 'steps': 2000, 'name': '[11,64,32,7] lr=0.05'},
    {'dims': [n_features, 128, 64, n_classes], 'lr': 0.01, 'steps': 2000, 'name': '[11,128,64,7] lr=0.01'},
]

hp_results = []

for cfg in configs:
    m = MLPScratch(
        layer_dims=cfg['dims'], learning_rate=cfg['lr'],
        n_steps=cfg['steps'], random_state=RANDOM_STATE,
        task='multiclass', activation_hidden='tanh'
    )
    m.fit(X_train, y_train, X_val=X_val, y_val=y_val, print_cost=False)
    
    # Değerlendirme
    train_acc = m.accuracy_history_train[-1]
    val_acc = m.accuracy_history_val[-1]
    y_p = m.predict(X_test)
    from sklearn.metrics import accuracy_score
    test_acc = accuracy_score(y_test.flatten(), y_p.flatten())
    
    hp_results.append({
        'Yapılandırma': cfg['name'],
        'Train Acc': f'{train_acc:.4f}',
        'Val Acc': f'{val_acc:.4f}',
        'Test Acc': f'{test_acc:.4f}',
        'Adım': cfg['steps']
    })
    print(f"{cfg['name']:30s} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

hp_df = pd.DataFrame(hp_results)
print('\n--- Hiperparametre Arama Sonuçları ---')
hp_df

---
## 8. Model 4: Scikit-learn MLPClassifier ile Karşılaştırma

Aynı mimari ve hiperparametrelerle **Scikit-learn MLPClassifier** kullanılarak model eğitilir.
Aynı eğitim/test seti ve SGD optimizasyon algoritması kullanılır.

In [ ]:
from sklearn.neural_network import MLPClassifier

# Scikit-learn MLPClassifier - Model 2 ile aynı mimari [64, 32]
sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='tanh',
    solver='sgd',
    learning_rate_init=0.01,
    max_iter=2000,
    random_state=RANDOM_STATE,
    verbose=False,
    early_stopping=False,
    batch_size=X_train.shape[0],  # Tam batch (full batch SGD)
    shuffle=False,
    n_iter_no_change=2000,
)

print('Scikit-learn MLPClassifier eğitiliyor...')
sklearn_mlp.fit(X_train, y_train.flatten())
print(f'Eğitim tamamlandı. Toplam iterasyon: {sklearn_mlp.n_iter_}')

### 8.1 Model 4 - Test Sonuçları

In [ ]:
y_pred_sklearn = sklearn_mlp.predict(X_test)
results_sklearn = print_metrics(y_test, y_pred_sklearn,
                                model_name='Model 4 (Scikit-learn MLPClassifier)',
                                task='multiclass')

plot_confusion_matrix(y_test, y_pred_sklearn,
                      model_name='Model 4 (Scikit-learn MLPClassifier)',
                      class_names=[f'K{i+3}' for i in range(n_classes)],
                      save_path='../results/wine_quality/model4_sklearn_cm.png')

### 8.2 Scikit-learn Eğitim Eğrisi

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
# Scikit-learn cost eğrisi
if hasattr(sklearn_mlp, 'loss_curve_'):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(sklearn_mlp.loss_curve_, color='#9C27B0', linewidth=2)
    ax.set_title('Scikit-learn MLPClassifier - Eğitim Kayıp Eğrisi', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../results/wine_quality/model4_sklearn_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 9. Model 5: PyTorch MLP ile Karşılaştırma

Aynı mimari ve hiperparametrelerle **PyTorch** kullanılarak model eğitilir.
SGD optimizasyon algoritması ve aynı eğitim/test seti kullanılır.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Cihaz ayari
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanilan cihaz: {device}')

# Tekrarlanabilirlik
torch.manual_seed(RANDOM_STATE)

# Veriyi tensor'a donustur
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train.flatten()).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.LongTensor(y_val.flatten()).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test.flatten()).to(device)

print(f'X_train_t shape: {X_train_t.shape}')
print(f'y_train_t shape: {y_train_t.shape}')

### 9.1 PyTorch Model Tanımı

In [ ]:
class PyTorchMLP(nn.Module):
    """PyTorch ile aynı mimariyi kullanan MLP modeli."""
    def __init__(self, n_input, n_hidden1, n_hidden2, n_output):
        super(PyTorchMLP, self).__init__()
        self.fc1 = nn.Linear(n_input, n_hidden1)
        self.fc2 = nn.Linear(n_hidden1, n_hidden2)
        self.fc3 = nn.Linear(n_hidden2, n_output)
        self.tanh = nn.Tanh()
        
        # Xavier initialization (tanh aktivasyonu ile uyumlu)
        nn.init.xavier_normal_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.zeros_(self.fc3.bias)
    
    def forward(self, x):
        x = self.tanh(self.fc1(x))
        x = self.tanh(self.fc2(x))
        x = self.fc3(x)  # CrossEntropyLoss softmax'ı içerir
        return x

# Model oluştur
pytorch_model = PyTorchMLP(n_features, 64, 32, n_classes).to(device)
print(pytorch_model)

### 9.2 PyTorch Eğitim

In [ ]:
# Eğitim ayarları
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(pytorch_model.parameters(), lr=0.01)

n_epochs = 2000
pytorch_train_losses = []
pytorch_val_losses = []
pytorch_train_accs = []
pytorch_val_accs = []

# Eğitim döngüsü
for epoch in range(n_epochs):
    pytorch_model.train()
    
    # İleri yayılım
    outputs = pytorch_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    # Geri yayılım 
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Kayıt
    pytorch_train_losses.append(loss.item())
    _, predicted = torch.max(outputs.data, 1)
    train_acc = (predicted == y_train_t).sum().item() / y_train_t.size(0)
    pytorch_train_accs.append(train_acc)
    
    # Validasyon
    pytorch_model.eval()
    with torch.no_grad():
        val_outputs = pytorch_model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t)
        _, val_predicted = torch.max(val_outputs.data, 1)
        val_acc = (val_predicted == y_val_t).sum().item() / y_val_t.size(0)
    pytorch_val_losses.append(val_loss.item())
    pytorch_val_accs.append(val_acc)
    
    if epoch % 200 == 0:
        print(f'Epoch {epoch:5d} | Train Loss: {loss.item():.6f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss.item():.6f} | Val Acc: {val_acc:.4f}')

print('-' * 70)
print(f'Epoch {n_epochs:5d} | Train Loss: {pytorch_train_losses[-1]:.6f} | Train Acc: {pytorch_train_accs[-1]:.4f} | Val Loss: {pytorch_val_losses[-1]:.6f} | Val Acc: {pytorch_val_accs[-1]:.4f}')

### 9.3 PyTorch - Eğitim Eğrileri

In [ ]:
os.makedirs('../results/wine_quality', exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(pytorch_train_losses, label='Train Loss', color='#2196F3', linewidth=2)
axes[0].plot(pytorch_val_losses, label='Val Loss', color='#FF5722', linewidth=2, linestyle='--')
axes[0].set_title('PyTorch MLP - Cost Eğrisi', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(pytorch_train_accs, label='Train Acc', color='#4CAF50', linewidth=2)
axes[1].plot(pytorch_val_accs, label='Val Acc', color='#FF9800', linewidth=2, linestyle='--')
axes[1].set_title('PyTorch MLP - Accuracy Eğrisi', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Model 5 (PyTorch MLP [11,64,32,7])', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('../results/wine_quality/model5_pytorch_egrileri.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.4 Model 5 - Test Sonuçları

In [ ]:
# PyTorch test değerlendirmesi
pytorch_model.eval()
with torch.no_grad():
    test_outputs = pytorch_model(X_test_t)
    _, y_pred_pytorch = torch.max(test_outputs.data, 1)
    y_pred_pytorch = y_pred_pytorch.cpu().numpy()

results_pytorch = print_metrics(y_test, y_pred_pytorch,
                                model_name='Model 5 (PyTorch MLP)',
                                task='multiclass')

plot_confusion_matrix(y_test, y_pred_pytorch,
                      model_name='Model 5 (PyTorch MLP [11,64,32,7])',
                      class_names=[f'K{i+3}' for i in range(n_classes)],
                      save_path='../results/wine_quality/model5_pytorch_cm.png')

---
## 10. Tüm Modellerin Karşılaştırılması ve Sonuçlar

In [ ]:
from src.metrics import compare_models

# Tüm sonuçları topla
all_results = {
    'M1: Scratch [11,32,7]': results_m1,
    'M2: Scratch [11,64,32,7]': results_m2,
    'M3: Scratch L2 Reg.': results_m3,
    'M4: Sklearn MLP': results_sklearn,
    'M5: PyTorch MLP': results_pytorch,
}

comparison_df = compare_models(all_results, save_path='../results/wine_quality/model_karsilastirma.png')

### 10.1 Model Seçimi

In [ ]:
# En iyi modeli seç
best_model_name = comparison_df['accuracy'].astype(float).idxmax()
best_accuracy = comparison_df.loc[best_model_name, 'accuracy']

print(f'\n🏆 En İyi Model: {best_model_name}')
print(f'   Test Accuracy: {float(best_accuracy):.4f} ({float(best_accuracy)*100:.2f}%)')
print(f'\n--- Model Seçim Kriterleri ---')
print(f'  - Test setinde en yüksek accuracy değerine sahip model seçilmiştir.')
print(f'  - Tüm modeller aynı train/test/val split, aynı random_state={RANDOM_STATE},') 
print(f'    aynı ölçeklendirme (StandardScaler) ve SGD optimizasyonu ile eğitilmiştir.')

### 10.2 Sonuç ve Yorum

Bu çalışmada Wine Quality veri seti üzerinde çoklu sınıflandırma problemi ele alınmıştır.

**Elde edilen bulgular:**

1. **Sıfırdan yazılan MLP** modeli başarıyla çoklu sınıflandırma yapabilmektedir (softmax + categorical cross entropy).
2. **Katman/nöron sayısı artırımı** model kapasitesini yükseltmiştir.
3. **L2 regülarizasyon** overfitting'i azaltmada etkili olmuştur.
4. Scikit-learn ve PyTorch kütüphaneleriyle aynı mimaride karşılaştırma yapılmıştır.
5. Wine Quality veri setindeki sınıf dengesizliği (kalite 5 ve 6 yoğunluklu), düşük temsil edilen sınıflarda performansı etkilemiştir.

**Gelecek çalışmalar:**
- Mini-batch SGD, momentum, Adam gibi farklı optimizasyon algoritmaları
- Batch normalizasyonu
- Daha agresif regülarizasyon (Dropout)
- Sınıf ağırlıklandırması (class weighting) ile dengesiz sınıfların ele alınması
- Veri artırma (data augmentation) teknikleri